In [1]:
#/root/jupyter_notebooks/mvm-accelerator/test.ipynb

import pynq
from pynq import Overlay, allocate, MMIO, PL
import time, re, threading
import numpy as np

### Config

In [2]:
DEBUG   = 1 # (0, 1, 2, 3)
PROFILE = 1 # (0, 1)

In [3]:
PROJ_DIR  = "/home/ubuntu/mvm-accelerator/hw/"
BIT       = PROJ_DIR + "design_1.bit"
MTRX_PATH = "/home/ubuntu/mvm-accelerator/trmult_reduced.bin"

The following values are fixed in hardware:

In [4]:
CDMA_BASE = 0xA0000000 
BRAM_BASE = 0x80000000 

AXIS_CLK_HZ = 200e6

NUM_ROWS = 16384
NUM_CHANNELS = 1 
ROWS_PER_CHANNEL = NUM_ROWS // NUM_CHANNELS

WORD_WIDTH = 64
ELEMENT_WIDTH = 64
ELEMENTS_PER_WORD = WORD_WIDTH // ELEMENT_WIDTH
BYTES_PER_WORD = WORD_WIDTH // 8

ELEMENTS_PER_ROW = 16384
WORDS_PER_ROW = ELEMENTS_PER_ROW // ELEMENTS_PER_WORD
BYTES_PER_ROW = WORDS_PER_ROW * BYTES_PER_WORD

NUM_PARTITIONS = NUM_CHANNELS

ELEMENTS_PER_PARTITION = ELEMENTS_PER_ROW // NUM_PARTITIONS
WORDS_PER_PARTITION = WORDS_PER_ROW // NUM_PARTITIONS
BYTES_PER_PARTITION = WORDS_PER_PARTITION * BYTES_PER_WORD

In [5]:
assert WORD_WIDTH % ELEMENT_WIDTH == 0
assert ELEMENTS_PER_ROW % NUM_CHANNELS == 0
assert NUM_ROWS % NUM_CHANNELS == 0

### Helpers

In [6]:
#https://github.com/iamNCJ/PYNQ-CDMA-Driver/blob/main/pynq_cdma/cdma.py

from pynq import DefaultIP
from pynq.buffer import PynqBuffer

class CDMA(DefaultIP):
    def __init__(self, description):
        super().__init__(description=description)

    bindto = ['xilinx.com:ip:axi_cdma:4.1']

    @property
    def idle(self):
        """
        `transfer` can only be called when the DMA is idle
        :return: True if the DMA engine is idle
        """
        return self.register_map.CDMASR.Idle

    def _do_transfer(self, source: int, dest: int, bytes_count: int):
        """
        Execute DMA transfer
        :param source: source address
        :param dest: destination address
        :param bytes_count: bytes to transfer
        :return: None
        """
        if not self.idle:
            raise Exception('DMA transfer can only start when engine is idle')

        assert source > 0
        if source > 0xFFFF_FFFF:
            self.register_map.SA_MSB[:] = source >> 8
        self.register_map.SA[:] = source & 0xFFFF_FFFF

        assert dest > 0
        if dest > 0xFFFF_FFFF:
            self.register_map.DA_MSB[:] = dest >> 8
        self.register_map.DA[:] = dest & 0xFFFF_FFFF

        assert bytes_count > 0
        self.register_map.BTT = bytes_count

        # Spin
        while not self.idle:
            pass

    def transfer(self, source, dest, bytes_count=None):
        """
        Transfer data through DMA
        :param source:
        :param dest:
        :param bytes_count:
        :return:
        """
        # Set source
        bytes_source = None
        if isinstance(source, PynqBuffer):
            source_addr = source.device_address
            bytes_source = source.nbytes
        elif isinstance(source, int):
            source_addr = source
        else:
            raise TypeError(f'Type {type(source)} not supported as source')

        bytes_dest = None
        if isinstance(dest, PynqBuffer):
            dest_addr = dest.device_address
            bytes_dest = dest.nbytes
        elif isinstance(dest, int):
            dest_addr = dest
        else:
            raise TypeError(f'Type {type(source)} not supported as dest')

        if bytes_count is None:
            if bytes_source is not None and bytes_dest is not None:
                bytes_count = min(bytes_source, bytes_dest)
            elif bytes_source is not None:
                bytes_count = bytes_source
            elif bytes_dest is not None:
                bytes_count = bytes_dest
            else:
                raise RuntimeError(f'Bytes to transfer is not set and cannot be inferred')

        self._do_transfer(source_addr, dest_addr, bytes_count)

In [7]:
from typing import Tuple, Dict, Any

try:
    from pynq import Overlay, MMIO
except ImportError:
    Overlay = None
    MMIO = None


class AxisDmaProfiler:
    """
    Address map:
    
        +0x00: STATUS [0]=have_result, [1]=running
        
        +0x04: CYCLES    [31:0]   +0x08: CYCLES    [63:32]
        +0x0C: BEATS     [31:0]   +0x10: BEATS     [63:32]
        +0x14: BYTES     [31:0]   +0x18: BYTES     [63:32]
        +0x1C: PKTS      [31:0]
        
        +0x20: GAP_LAST  [31:0]   +0x24: GAP_LAST  [63:32]
        +0x28: GAP_TOTAL [31:0]   +0x2C: GAP_TOTAL [63:32]
        +0x30: GAP_MIN   [31:0]   +0x34: GAP_MIN   [63:32]
        +0x38: GAP_MAX   [31:0]   +0x3C: GAP_MAX   [63:32]
        +0x40: GAP_COUNT [31:0]
        
        +0x44: GAP_STATUS [0]=gap_valid, [1]=armed
        
        +0x48: BUSY_TOTAL[31:0]   +0x4C: BUSY_TOTAL[63:32]
    """
    
    OFF_STATUS = 0x00
    OFF_CYCLES = 0x04
    OFF_BEATS  = 0x0C
    OFF_BYTES  = 0x14
    OFF_PKTS   = 0x1C
    
    OFF_GAP_LAST   = 0x20
    OFF_GAP_TOTAL  = 0x28
    OFF_GAP_MIN    = 0x30
    OFF_GAP_MAX    = 0x38
    OFF_GAP_COUNT  = 0x40
    OFF_GAP_STATUS = 0x44
    OFF_BUSY_TOTAL = 0x48
    
    CH_WINDOW = 0x100 # 0x100 bytes per channel

    def __init__(self, overlay, channels=NUM_CHANNELS, ip_name_contains="axis_dma_profiler"):
        self.channels = channels
        self.overlay = overlay

        key = next((k for k in overlay.ip_dict if ip_name_contains in k), None)
        
        if key is None:
            raise ValueError(
                f"Could not find IP containing '{ip_name_contains}' in overlay.ip_dict. "
                f"Pass base=... explicitly or adjust ip_name_contains."
            )
        ip = overlay.ip_dict[key]
        base = int(ip["phys_addr"])
        size = int(ip["addr_range"])

        if base is None: raise ValueError("Profiler base address is unknown.")
        if size is None: raise ValueError("Profiler size address is unknown.")

        self.base = base
        self.size = size
        self.mmio = MMIO(self.base, self.size)
        
        if DEBUG: print(f"    {self.__repr__()}")

    def _rd32(self, off):
        return self.mmio.read(off)

    def _rd64_stable(self, off, attempts=4):
        for _ in range(attempts):
            hi1 = self._rd32(off + 4)
            lo  = self._rd32(off)
            hi2 = self._rd32(off + 4)
            if hi1 == hi2:
                return (hi1 << 32) | lo
        return -1

    def _ch_base(self, ch: int) -> int:
        if not (0 <= ch < self.channels):
            raise ValueError(f"Channel out of range: {ch} (channels={self.channels})")
        return ch * self.CH_WINDOW

    # ========== Public ==========

    def stats(self, ch) -> Dict[str, int]:
        """
        Reads (cycles, beats, bytes, pkts). Prefer calling when have_result=1 and running=0.
        """
        b = self._ch_base(ch)
        cycles = self._rd64_stable(b + self.OFF_CYCLES)
        beats  = self._rd64_stable(b + self.OFF_BEATS)
        bytes_ = self._rd64_stable(b + self.OFF_BYTES)
        pkts   = self._rd32(b + self.OFF_PKTS)
        
        return {
            "cycles": cycles, 
            "beats":  beats, 
            "bytes":  bytes_, 
            "pkts":   pkts
        }
    
    def gap_stats(self, ch) -> Dict[str, Any]:
        """
        Inter-packet gap metrics for channel ch.
        """
        b = self._ch_base(ch)
        gap_last   = self._rd64_stable(b + self.OFF_GAP_LAST)
        gap_total  = self._rd64_stable(b + self.OFF_GAP_TOTAL)
        gap_min    = self._rd64_stable(b + self.OFF_GAP_MIN)
        gap_max    = self._rd64_stable(b + self.OFF_GAP_MAX)
        gap_count  = self._rd32(b + self.OFF_GAP_COUNT)
        busy_total = self._rd64_stable(b + self.OFF_BUSY_TOTAL)

        return {
            "gap_last":   gap_last,
            "gap_total":  gap_total,
            "gap_min":    gap_min,
            "gap_max":    gap_max,
            "gap_count":  gap_count,
            "busy_total": busy_total
        }

    def throughput(self, ch, axis_clk_hz=AXIS_CLK_HZ) -> Tuple[float, float, Dict[str, int]]:
        """
        Returns (gbps, handshake_utilization, stats) for a completed frame on channel ch.
        - gbps  = (bytes * 8 / cycles) * axis_clk_hz / 1e9
        - handshake_utilization = beats / cycles   (fraction of cycles that handshook)
        """
        s = self.stats(ch)
        cycles = max(1, s["cycles"])
        gbps = (s["bytes"] * 8.0 * axis_clk_hz / cycles) / 1e9
        util = s["beats"] / cycles
        return gbps, util, s
    
    def gap_summary(self, ch, axis_clk_hz=AXIS_CLK_HZ):
        """
        Analogous to `throughput`: return (g, mean_gap_ns, util_long_run).
          - g: dict with gap aggregates (gap_last, gap_min/max, gap_total, gap_count, busy_total)
          - mean_gap_ns: average inter-packet gap in nanoseconds
          - util_long_run: busy_total / (busy_total + gap_total)
        """
        g = self.gap_stats(ch)
        gap_total  = g["gap_total"]
        gap_count  = g["gap_count"]
        busy_total = g["busy_total"]

        mean_gap_cycles = (gap_total / gap_count) if gap_count else 0.0
        mean_gap_ns     = (1e9 * mean_gap_cycles / axis_clk_hz) if axis_clk_hz else 0.0
        util_long_run   = (busy_total / (busy_total + gap_total)) if (busy_total + gap_total) else 0.0

        return g, int(mean_gap_cycles), int(mean_gap_ns), util_long_run

    def __repr__(self) -> str:
        return f"AxisDmaProfiler(base=0x{self.base:08X}, size=0x{self.size:X}, channels={self.channels})"


In [8]:
def allocate_buf(dim1, dim2=None):
    if dim2 is not None:
        return allocate(shape=(dim1, dim2), dtype=np.float64, cacheable=False)
    else:
        return allocate(shape=(dim1,), dtype=np.float64, cacheable=False)

In [9]:
def write_vec(overlay, vector):
    if DEBUG: print("Writing vector to BRAM.")
    cdma = CDMA(overlay.ip_dict['axi_cdma_0'])

    # Dummy vector
    for i in range(NUM_PARTITIONS):
        vector[i, :] = np.arange(
            i * ELEMENTS_PER_PARTITION + 1,
            (i + 1) * ELEMENTS_PER_PARTITION + 1,
            dtype=np.float64) / 10000.0

    for i in range(NUM_PARTITIONS):
        dest = BRAM_BASE + i * BYTES_PER_PARTITION
        if DEBUG: print(f"    vec[{i}]: src_addr=0x{vector[i].physical_address:8x}, dest_addr=0x{dest:8x}")
        cdma.transfer(vector[i], dest)

In [10]:
def get_dmas(overlay):
    dmas = [
        getattr(overlay, name)
        for name in sorted(overlay.ip_dict, key=lambda n: int(re.search(r"\d+$", n).group()))
        if name.startswith("axi_dma")
    ]
    if DEBUG: print("    dmas", [d.description['fullpath'] for d in dmas])
    if len(dmas) != NUM_CHANNELS:
        raise RuntimeError(f"Check overlay. Found {len(dmas)} DMAs with NUM_CHANNELS={NUM_CHANNELS}.")
    return dmas

In [11]:
def create_workers(dmas, matrix):
    if DEBUG: print("Creating threads.")

    threads = []
    for ch in range(NUM_CHANNELS):
        t = threading.Thread(
            target=worker,
            args=(ch, dmas[ch], matrix[ch]),
            daemon=True
        )
        threads.append(t)
        if DEBUG: print(f"    Initialized worker {ch}")
    return threads

In [12]:
def worker(ch, dma, matrix_chunk):
    if DEBUG > 1: print(f"[WORKER {ch}] Starting up...")
        
    dma.sendchannel.transfer(matrix_chunk)
    dma.sendchannel.wait()
    
    """
    for i in range(0, ROWS_PER_CHANNEL, ring_depth):
        block = matrix_chunk[i:i+ring_depth]
        base_row = ch * ROWS_PER_CHANNEL + i
        
        for j, row in enumerate(block):
            dma.sendchannel.transfer(row)
            if DEBUG > 1: print(f"[WORKER {ch}] queued row {base_row+j}")
        
        dma.sendchannel.wait()
        if DEBUG > 1: print(f"[WORKER {ch}] block {i//ring_depth} complete")
    """

### Main

In [14]:
if __name__ == "__main__":
    if DEBUG: 
        print(f"\n================================ Parameters =================================")
        
        print(f"""
    CDMA_BASE = 0x{CDMA_BASE:8X}
    BRAM_BASE = 0x{BRAM_BASE:8X}

    NUM_ROWS         = {NUM_ROWS}
    NUM_CHANNELS     = {NUM_CHANNELS} 
    ROWS_PER_CHANNEL = {ROWS_PER_CHANNEL}

    WORD_WIDTH        = {WORD_WIDTH}
    ELEMENT_WIDTH     = {ELEMENT_WIDTH}
    ELEMENTS_PER_WORD = {ELEMENTS_PER_WORD}
    BYTES_PER_WORD    = {BYTES_PER_WORD}

    ELEMENTS_PER_ROW = {ELEMENTS_PER_ROW}
    WORDS_PER_ROW    = {WORDS_PER_ROW}
    BYTES_PER_ROW    = {BYTES_PER_ROW}

    NUM_PARTITIONS = {NUM_PARTITIONS}

    ELEMENTS_PER_PARTITION = {ELEMENTS_PER_PARTITION}
    WORDS_PER_PARTITION    = {WORDS_PER_PARTITION}
    BYTES_PER_PARTITION    = {BYTES_PER_PARTITION}""")

    PL.reset()
    matrix = vector = result = None 

    try:
        if DEBUG: print("\n================================ Initialize =================================\n")

        # Load overlay
        if DEBUG: print("Loading Overlay.")
        overlay = Overlay(BIT, download=True)
        if not overlay.is_loaded():
            raise RuntimeError("Overlay download failed.")
        
        # Get DMAs and Profiler
        dmas = get_dmas(overlay)
        if PROFILE: prof = AxisDmaProfiler(overlay)

        # Allocate buffers
        if DEBUG: print("Allocating buffers.")
        matrix = [allocate_buf(ROWS_PER_CHANNEL, ELEMENTS_PER_ROW) for _ in range(NUM_CHANNELS)]
        vector =  allocate_buf(NUM_PARTITIONS, ELEMENTS_PER_PARTITION)
        result = [allocate_buf(ROWS_PER_CHANNEL) for _ in range(NUM_CHANNELS)]
        if DEBUG:
            [print(f"    matrix[{i}]: src_addr=0x{m.physical_address:8X}") for i,m in enumerate(matrix)]
            print(f"    vector: src_addr=0x{vector.physical_address:8X}")
            [print(f"    result[{i}]: src_addr=0x{r.physical_address:8X}") for i,r in enumerate(result)]

        # Read matrix into buffer
        if DEBUG: print("Reading matrix file.")
        with open(MTRX_PATH, "rb") as a_file:
            for i in range(NUM_CHANNELS):
                for j in range(ROWS_PER_CHANNEL):
                    view = memoryview(matrix[i][j]).cast("B")
                    if a_file.readinto(view) != BYTES_PER_ROW:
                        raise RuntimeError(f"Failed to read full row {i}.")
                    if DEBUG > 2: print(f"    matrix[{i}][{j}]: {matrix[i][j]}")

        # Write vector to BRAM
        t0_vec = time.perf_counter()
        write_vec(overlay, vector) 
        t1_vec = time.perf_counter()

        # Create worker threads
        threads = create_workers(dmas, matrix)

        # Execute
        if DEBUG: print("\n================================== Compute ==================================\n")
        
        t0_comp = time.perf_counter()

        for ch in range(NUM_CHANNELS):
            dmas[ch].recvchannel.transfer(result[ch])

        for t in threads: t.start()
        for t in threads: t.join()

        for ch in range(NUM_CHANNELS):
            dmas[ch].recvchannel.wait()

        t1_comp = time.perf_counter()

        # Print results
        if DEBUG:
            for i in range(NUM_CHANNELS):
                for j in range(ROWS_PER_CHANNEL):
                    row_idx = i * ROWS_PER_CHANNEL + j
                    if (row_idx > 5 and row_idx < NUM_ROWS-5):
                        if not DEBUG > 2: # Only print all rows if DEBUG == 3
                            if (row_idx == 6): print("...")
                            continue
                    actual = result[i][j]
                    expected = float(np.dot(matrix[i][j], vector.flatten()))
                    print(f"Row {row_idx}: Actual={actual:.32f} | Expected={expected:.32f}")
            
            print("\n================================ Performance ================================\n")
            
            if PROFILE:
                for i in range(NUM_CHANNELS):
                    gbps, util, s = prof.throughput(ch=i)
                    g, mean_gap_cycles, mean_gap_ns, util_long_run = prof.gap_summary(ch=i)
                    print(f"Channel {i}:")
                    if DEBUG > 1:
                        print(f"    {s}")
                        print(f"    {g}")
                    print(f"    Throughput: {gbps:.3f} Gbps | Handshake util: {util:.2%}")
                    print(f"    Avg gap length: {mean_gap_cycles} cycles, {mean_gap_ns} ns | Busy util: {util_long_run:.2%}")
        
        print(f"\nOverhead time: {t1_vec - t0_vec:.6f}s")
        print(f"Compute time:  {t1_comp - t0_comp:.6f}s")

    finally:
        for buf in [matrix, result]:
            if buf is not None:
                for b in buf: b.freebuffer()
        if vector is not None: vector.freebuffer()


================================ Parameters =================================

    CDMA_BASE = 0xA0000000
    BRAM_BASE = 0x80000000

    NUM_ROWS         = 1024
    NUM_CHANNELS     = 1 
    ROWS_PER_CHANNEL = 1024

    WORD_WIDTH        = 64
    ELEMENT_WIDTH     = 64
    ELEMENTS_PER_WORD = 1
    BYTES_PER_WORD    = 8

    ELEMENTS_PER_ROW = 8192
    WORDS_PER_ROW    = 8192
    BYTES_PER_ROW    = 65536

    NUM_PARTITIONS = 1

    ELEMENTS_PER_PARTITION = 8192
    WORDS_PER_PARTITION    = 8192
    BYTES_PER_PARTITION    = 65536

================================ Initialize =================================

Loading Overlay.
    dmas ['axi_dma_0']
    AxisDmaProfiler(base=0xA0040000, size=0x1000, channels=1)
Allocating buffers.
    matrix[0]: src_addr=0x3D300000
    vector: src_addr=0x375B0000
    result[0]: src_addr=0x375C0000
Reading matrix file.
Writing vector to BRAM.
    vec[0]: src_addr=0x375b0000, dest_addr=0x80000000
Creating threads.
    Initialized worker 0

===============